<a href="https://colab.research.google.com/github/ikeyareinaldosae/ITC501---Artificial-Intelligence-Applications/blob/main/Week_2_FeatureTools_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 2 Lab (Task 2) - Data preparation using FeatureTools**

## Mount the google drive

In [5]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive') # Only run this if needed

# Construct the path to the dataset
data_folder = "/content/drive/MyDrive/datasets"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Overview

We will be using the "House Prices - Advanced Regression Techniques" dataset available on Kaggle. This dataset challenges us to predict the final price of each home based on 79 explanatory variables covering a wide range of aspects related to residential homes in Ames, Iowa. Link [here](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)

### Install Featuretools:
We'll start by installing the `featuretools` library, which will help us automate some feature engineering tasks

In [2]:
!pip install featuretools --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.9/587.9 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.2/215.2 kB 17.7 MB/s eta 0:00:00


### Import Necessary Libraries
Next, we import two essential libraries:
- `featuretools` (aliased as `ft`) for automated feature engineering.
- `pandas` for data manipulation and analysis.

In [3]:
import pandas as pd
import featuretools as ft

### 3. Load the Dataset
Ensure your Google Drive is mounted and then construct the path to the dataset. We'll load the dataset into a pandas DataFrame.

In [6]:
# Load the dataset
data = pd.read_csv(f'{data_folder}/house_prices.csv')

data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Using FeatureTools

### Create an EntitySet
An `EntitySet` is a collection of tables (entities) and the relationships between them. Let's create an empty `EntitySet` named 'HousePrices'.


In [14]:
# Create an EntitySet
es = ft.EntitySet(id='HousePrices')

### Add the Dataset as an Entity

We add the "House Prices - Advanced Regression Techniques" dataset as an entity to the `EntitySet`. In this dataset, each row represents information about a residential home. We'll use the 'Id' column as the index, as it serves as a unique identifier for each home.

In [15]:
# Add the entire data as an entity ('Id' is the unique identifier)
es.add_dataframe(dataframe_name='HousePrices', dataframe=data, index='Id')

/usr/local/lib/python3.12/dist-packages/featuretools/entityset/entityset.py:724: UserWarning: A Woodwork-initialized DataFrame was provided, so the following parameters were ignored: index
  warnings.warn(


Entityset: HousePrices
  DataFrames:
    HousePrices [Rows: 1460, Columns: 81]
  Relationships:
    No relationships

### 6. Run Deep Feature Synthesis (DFS)
Deep Feature Synthesis (DFS) is a method to generate new features by applying a set of feature primitives to the entities in an `EntitySet`. We'll run DFS to automatically generate new features. The `max_depth` parameter controls the depth of the relationships to be explored.

In our case, we call `ft.dfs` with the following parameters:
- `entityset`: The EntitySet we created, which holds our data.
- `target_dataframe_name`: The name of the dataframe on which to make feature calculations.
- `trans_primitives`: A list of transformation primitives to be used for generating new features. Transformation primitives are functions that generate new features by operating on existing data in a single table. In our scenario, we've chosen `['multiply_numeric', 'divide_numeric', 'bin']` as the transformation primitives to generate interaction terms and bin continuous variables, which can help capture more complex relationships in the data.
- `max_depth`: This parameter controls the maximum depth of the features being generated. A depth of 3 means it'll stack up to three primitives together to create new features.

With these parameters, `ft.dfs` will generate new features based on the specified transformation primitives, exploring relationships up to the specified maximum depth.

What primitives can we use? Lots! Over 200 of them

```
ft.primitives.list_primitives()
```

In [16]:
ft.primitives.list_primitives()

,name,type,description,valid_inputs,return_type
0,n_most_common_frequency,aggregation,Determines the frequency of the n most common ...,<ColumnSchema (Semantic Tags = ['category'])>,<ColumnSchema (Logical Type = Categorical) (Se...
1,num_consecutive_less_mean,aggregation,Determines the length of the longest subsequen...,<ColumnSchema (Semantic Tags = ['numeric'])>,<ColumnSchema (Logical Type = IntegerNullable)...
2,last,aggregation,Determines the last value in a list.,<ColumnSchema>,None
3,trend,aggregation,Calculates the trend of a column over time.,"<ColumnSchema (Semantic Tags = ['numeric'])>, ...",<ColumnSchema (Semantic Tags = ['numeric'])>
4,std,aggregation,Computes the dispersion relative to the mean v...,<ColumnSchema (Semantic Tags = ['numeric'])>,<ColumnSchema (Semantic Tags = ['numeric'])>
...,...,...,...,...,...
198,savgol_filter,transform,Applies a Savitzky-Golay filter to a list of v...,<ColumnSchema (Semantic Tags = ['numeric'])>,<ColumnSchema (Logical Type = Double) (Semanti...
199,total_word_length,transform,Determines the total word length.,<ColumnSchema (Logical Type = NaturalLanguage)>,<ColumnSchema (Logical Type = IntegerNullable)...
200,cum_max,transform,Calculates the cumulative maximum.,<ColumnSchema (Semantic Tags = ['numeric'])>,<ColumnSchema (Semantic Tags = ['numeric'])>
201,num_words,transform,Determines the number of words in a string. Wo...,<ColumnSchema (Logical Type = NaturalLanguage)>,<ColumnSchema (Logical Type = IntegerNullable)...


In [17]:
# Specify selected transformation primitives
trans_primitives = ['multiply_numeric', 'divide_numeric']

# Run Deep Feature Synthesis (DFS) with the specified transformation primitives
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='HousePrices',
    trans_primitives=trans_primitives,
    max_depth=1
)



The function returns two objects:
- `feature_matrix`: A DataFrame that holds the original data and the new features.
- `feature_defs`: A list of feature definitions.

### 7. Explore Generated Features:
`feature_matrix` now contains the original data along with new features generated by DFS. Let's take a look at some of the generated features.

In [18]:
feature_matrix.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,TotalBsmtSF * WoodDeckSF,TotalBsmtSF * YearBuilt,TotalBsmtSF * YearRemodAdd,TotalBsmtSF * YrSold,WoodDeckSF * YearBuilt,WoodDeckSF * YearRemodAdd,WoodDeckSF * YrSold,YearBuilt * YearRemodAdd,YearBuilt * YrSold,YearRemodAdd * YrSold
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0.0,1714568.0,1714568.0,1718848.0,0.0,0.0,0.0,4012009.0,4022024.0,4022024.0
2,20,RL,80,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,376076.0,2493712.0,2493712.0,2532834.0,588848.0,588848.0,598086.0,3904576.0,3965832.0,3965832.0
3,60,RL,68,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0.0,1840920.0,1841840.0,1847360.0,0.0,0.0,0.0,4006002.0,4018008.0,4020016.0
4,70,RL,60,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0.0,1447740.0,1489320.0,1516536.0,0.0,0.0,0.0,3772550.0,3841490.0,3951820.0
5,60,RL,84,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,219840.0,2290000.0,2290000.0,2299160.0,384000.0,384000.0,385536.0,4000000.0,4016000.0,4016000.0


In [19]:
len(feature_defs)

2077

It created over 2000 features! Is this reasonable?

# Your Turn

Consider the generated features from above. Try using different primatives and depths and see how many features and what features are created.

Are they reasonable? How many are created?

Consider the time saving vs the number created.

In [20]:
# Drop features that have a high number of NaN/Inf values or zero variance
from featuretools.selection import remove_low_information_features

feature_matrix, feature_defs = remove_low_information_features(feature_matrix, feature_defs)

feature_matrix.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,TotalBsmtSF * WoodDeckSF,TotalBsmtSF * YearBuilt,TotalBsmtSF * YearRemodAdd,TotalBsmtSF * YrSold,WoodDeckSF * YearBuilt,WoodDeckSF * YearRemodAdd,WoodDeckSF * YrSold,YearBuilt * YearRemodAdd,YearBuilt * YrSold,YearRemodAdd * YrSold
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0.0,1714568.0,1714568.0,1718848.0,0.0,0.0,0.0,4012009.0,4022024.0,4022024.0
2,20,RL,80,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,376076.0,2493712.0,2493712.0,2532834.0,588848.0,588848.0,598086.0,3904576.0,3965832.0,3965832.0
3,60,RL,68,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0.0,1840920.0,1841840.0,1847360.0,0.0,0.0,0.0,4006002.0,4018008.0,4020016.0
4,70,RL,60,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0.0,1447740.0,1489320.0,1516536.0,0.0,0.0,0.0,3772550.0,3841490.0,3951820.0
5,60,RL,84,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,219840.0,2290000.0,2290000.0,2299160.0,384000.0,384000.0,385536.0,4000000.0,4016000.0,4016000.0


# Submission
When you are ready to submit please go to `File` -> `Print` -> `Print PDF` and then upload and submit in the Campus Online.

Please save your file to GitHub via: `File` -> `Save a copy in GitHub`